# M5 — Skills + SODA quality checks
**Duration:** 30 min | **Participant notebook:** `M5_SODA.ipynb`

---

## At a glance

| | |
|---|---|
| **Copilot skill taught** | `/add-soda-checks` skill (project-agnostic), three levels of reusability |
| **Key files** | `.github/skills/add-soda-checks.md`, `soda_checks/curve_quality.yml` |
| **What participants produce** | `soda_checks/curve_quality.yml` (generated by `/add-soda-checks`); `run_soda_checks()` implemented in `validator.py` using `soda-core-duckdb` |

**Prerequisite:** `curve_pipeline/validator.py` with `validate_record` must be complete and green (M4). `run_soda_checks` does not exist yet — that is what M5 adds.

**The module is 30 min by design.** One skill invocation (YAML), one agent prompt (implementation), two data runs, a debrief.

## Key teaching moment

**The difference between a prompt and a skill is whether it knows your project.**

Deliver this at the three-level table:

> "A `.github/prompts/` file knows your column names, your file paths, your CET rules — it is project-specific by design. A `.github/skills/` file reads whatever you point it at and adapts. You could copy `/add-soda-checks` to any pipeline repo and it would work. That is the difference."

Secondary teaching moment — what SODA can't catch:

> "The regression runner checks that `cetStarttime` has the right *value*. SODA checks that it is *present*. Ask the room: would a `missing_count = 0` check on `cetStarttime` have caught the copy-through bug? The answer is no — the value was present, just wrong. That is why both tools exist."

## Questions and answers

Q: "Could you use `/add-soda-checks` on a different pipeline with different column names?"

A: Yes — that is the point of a skill. It reads the schema from the loader file you point it at. The only thing that needs to change is the `#file:` reference in the prompt. The skill itself is repo-agnostic.

---

Q: "The SODA check for `missing_count` on `cetStarttime` passes on the buggy input. Why?"

A: Because the copy-through bug *copies* startTime — the value is present, just wrong. SODA only checks presence and type, not correctness of computed values. That is the regression runner's job.

---

Q: "In M3 we wrote a custom prompt. In M5 we used a skill. What was the difference?"

A: The prompt in M3 knew the project — it referenced `cet_converter.py` and the specific column names. The skill in M5 discovers the schema at runtime. Prompts are faster to write for known tasks; skills are more reusable across projects.

## Watch for

**SODA check includes `min > 0` on `quote`.**

What it looks like: The generated YAML has a numeric range check that rejects negative quotes.

Response: Stop them. "Are negative prices valid in power markets?" (Yes.) "So what does this check do to a valid record?" Delete the range check and keep only `is_numeric`.

---

**`curve_with_issues.csv` shows 0 failures.**

What it looks like: Both the clean and issues datasets return all-zero counts.

Response: The validator's `run_soda_checks()` method may not be wired to the YAML yet, or the YAML checks don't match the actual column names. Ask: "What column names does the YAML use? Are they the same as the CSV headers?"

---

**Participants finish early.**

This module is short. Have a natural extension ready: "Try `/add-soda-checks` on `regression/regression_dc_fwd_power.py` instead of the loader. What checks does it generate? Are they the same?"
---

**`run_soda_checks` fails with a SQL error on the `order` column.**

What it looks like: DuckDB raises a parser error — `order` is a reserved SQL word.

Response: The implementation must rename the column before loading into DuckDB: `df = df.rename(columns={"order": "curve_order"})`. The YAML must also use `curve_order` for that check. The Step 2 agent prompt already includes this — if it happened, the agent didn't follow the prompt closely enough.

## Timing notes

| Segment | Planned | Notes |
|---|---|---|
| Three-level reusability intro | 5 min | Spend time on the prompt vs skill distinction |
| SODA intro + Step 1 `/add-soda-checks` | 8 min | Most participants get the YAML on first try |
| Step 2 — implement `run_soda_checks` | 7 min | Agent prompt is short; wiring takes most of the time |
| Step 3 — run clean + issues data | 7 min | Debugging if validator incomplete |
| Debrief | 3 min | "Would SODA have caught the bug?" |

**Total: 30 min.**

**If M4 ran long and `validator.py` is not complete:** Skip Steps 2–3 entirely. Do only Step 1 (generate the YAML) — that is the skill demonstration. Participants can wire `run_soda_checks` independently after the session.